In [ ]:
!nvidia-smi

# GSM8K 0-Shot Evaluation — LLaMA-3 8B (vLLM)

In [ ]:
!pip install "protobuf==5.29.5"
!pip install -q vllm datasets scikit-learn

In [ ]:
from huggingface_hub import login
login()

In [ ]:
import os
import re
import json
import time
import numpy as np
from datasets import load_dataset

print("Loading GSM8K test set...")
ds = load_dataset("gsm8k", "main", split="test")

print(f"Dataset size: {len(ds)} examples")
print(f"Columns: {ds.column_names}")


def extract_answer_number(text):
    """Extract the last number from text. Handles GSM8K '####' marker and commas."""
    if "####" in text:
        text = text.split("####")[-1]
    text = text.replace(",", "")
    matches = re.findall(r"-?\d+\.?\d*", text)
    return matches[-1] if matches else None

In [ ]:
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
MAX_NEW_TOKENS = 512
EVAL_SIZE = len(ds)

print({
    "model": MODEL_NAME,
    "eval_size": EVAL_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
})

# Model

In [ ]:
if __name__ == '__main__':
    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams

    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

    print(f"Loading {MODEL_NAME} with vLLM...")
    llm = LLM(
        model=MODEL_NAME,
        dtype="float16",
        gpu_memory_utilization=0.95,
        max_model_len=4096,
        enforce_eager=False,
    )

    sampling_params = SamplingParams(
        temperature=0,
        top_k=20,
        max_tokens=MAX_NEW_TOKENS,
    )

    print("Model loaded successfully!")

# Output Directory

In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_SAVE_DIR = "/content/drive/MyDrive/Colab_Data/GSM8K/Llama3_8B_GSM8K_Eval_vLLM"
except Exception:
    DRIVE_SAVE_DIR = os.path.abspath("./llama3_8b_gsm8k_outputs")

os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"Saving results to: {DRIVE_SAVE_DIR}")

# Checkpoint Setup & Prompt Construction

In [ ]:
CHECKPOINT_FILE = os.path.join(DRIVE_SAVE_DIR, "llama3_8b_gsm8k_vllm_checkpoint.jsonl")
OUTPUTS_CACHE   = os.path.join(DRIVE_SAVE_DIR, "llama3_8b_gsm8k_vllm_raw_outputs.json")

results = []
if os.path.exists(CHECKPOINT_FILE):
    print(f"Found checkpoint: {CHECKPOINT_FILE}")
    with open(CHECKPOINT_FILE) as f:
        for line in f:
            try:
                results.append(json.loads(line))
            except json.JSONDecodeError:
                continue
    print(f"Loaded {len(results)} previously evaluated examples.")
else:
    print("No checkpoint found — starting from scratch.")

subset = ds.select(range(EVAL_SIZE))
questions = subset["question"]
ground_truths = [extract_answer_number(a) for a in subset["answer"]]

already_done = len(results)
remaining_qs = questions[already_done:]
print(f"{already_done} already done, {len(remaining_qs)} remaining.")

In [ ]:
prompts = [
    tokenizer.apply_chat_template(
        [{"role": "user", "content": q}],
        tokenize=False,
        add_generation_prompt=True,
    )
    for q in remaining_qs
]

print(f"Built {len(prompts)} prompts.")
if prompts:
    print(f"\nExample prompt (first 400 chars):\n{prompts[0][:400]}")
else:
    print("All requested examples are already processed.")

# Generation

In [ ]:
if prompts:
    print(f"Generating responses for {len(prompts)} prompts...")

    gen_start = time.time()
    outputs = llm.generate(prompts, sampling_params)
    gen_time = time.time() - gen_start

    total_new_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
    throughput = total_new_tokens / gen_time if gen_time > 0 else None

    print("\nGeneration complete.")
    print(f"  Time:       {gen_time / 60:.1f} min")
    print(f"  Tokens:     {total_new_tokens:,}")
    if throughput:
        print(f"  Throughput: {throughput:.1f} tokens/sec")

    raw_cache = [
        {
            "question_idx": already_done + i,
            "question": remaining_qs[i],
            "ground_truth": ground_truths[already_done + i],
            "response": o.outputs[0].text,
            "n_tokens": len(o.outputs[0].token_ids),
        }
        for i, o in enumerate(outputs)
    ]

    with open(OUTPUTS_CACHE, "w") as f:
        json.dump(raw_cache, f)
    print(f"Raw outputs cached to: {OUTPUTS_CACHE}")
else:
    print("No prompts to generate — loading cached outputs.")
    if os.path.exists(OUTPUTS_CACHE):
        with open(OUTPUTS_CACHE) as f:
            raw_cache = json.load(f)
    else:
        raw_cache = []
    gen_time = None
    throughput = None
    total_new_tokens = sum(r.get("n_tokens", 0) for r in raw_cache)

print(f"Raw items available for scoring: {len(raw_cache)}")

# Scoring & Final Metrics

In [ ]:
new_results = []
for item in raw_cache:
    idx = item["question_idx"]
    response = item["response"]

    pred = extract_answer_number(response)
    gt = ground_truths[idx]

    is_correct = False
    if pred is not None and gt is not None:
        try:
            is_correct = float(pred) == float(gt)
        except ValueError:
            pass

    res = {
        "question_idx": idx,
        "prediction": pred,
        "ground_truth": gt,
        "raw_response": response.strip(),
        "is_correct": is_correct,
        "is_unknown": pred is None,
        "new_tokens": item["n_tokens"],
    }
    new_results.append(res)

if new_results:
    with open(CHECKPOINT_FILE, "a") as f:
        for r in new_results:
            f.write(json.dumps(r) + "\n")
    results.extend(new_results)

print(f"Total scored results: {len(results)}")

correct_count = sum(1 for r in results if r["is_correct"])
unknown_count = sum(1 for r in results if r["is_unknown"])
all_new_tokens = sum(r["new_tokens"] for r in results)
accuracy = correct_count / len(results) if results else 0

known_preds = [r for r in results if not r["is_unknown"]]
accuracy_known = (
    sum(1 for r in known_preds if r["is_correct"]) / len(known_preds)
    if known_preds
    else 0
)

final_metrics = {
    "method": "0_shot_vllm",
    "model": MODEL_NAME,
    "dataset": "gsm8k/main:test",
    "eval_size": len(results),
    "accuracy": accuracy,
    "accuracy_known_only": accuracy_known if known_preds else "N/A",
    "unknown_frac": unknown_count / len(results) if results else 0,
    "total_new_tokens": all_new_tokens,
    "gen_tokens_per_sec": throughput if throughput is not None else "N/A (loaded from cache)",
    "total_gen_time_min": gen_time / 60 if gen_time is not None else "N/A (loaded from cache)",
}

print("\n=== Final Metrics ===")
for k, v in final_metrics.items():
    print(f"  {k}: {v}")

METRICS_FILE = os.path.join(DRIVE_SAVE_DIR, "final_metrics.json")
with open(METRICS_FILE, "w") as f:
    json.dump(final_metrics, f, indent=2)
print(f"\nMetrics saved to: {METRICS_FILE}")